# Option C — two-detector vessel screening

**Detector A** = pretrained `yolo11s_tci.pt` on 8-bit RGB (learned vessel *shape*).
**Detector B** = adaptive NIR bright-object threshold on dark water (uses *contrast*).

They fail differently, which is the point: B independently sweeps for anything A missed, so we
can certify a scene as **clean** — one dominant vessel, nothing else nearby. See
`option_c_detection_plan.md` for the why.

This notebook is written to run on **two kinds of input**:

| | development (today) | production (Thursday) |
|---|---|---|
| source | Sentinel-2 L2A `.SAFE` | PlanetScope `ortho_analytic_4b_sr` |
| GSD | 10 m | 3 m |
| rendered RGB | `TCI_10m.jp2` | `ortho_visual` |
| reflectance bands | `B02/B03/B04/B08` | bands 1–4 of the analytic asset |

Both go through **one loader seam** (§3) that returns the same `Scene` object, so switching to
PlanetScope is a config change, not a rewrite.

---
### Stage 1 (this file so far)
§0 config · §1 inventory · §2 pick a window · §3 load · §4 derive RGB · §5 detector A

Still to come: water mask, detector B, fusion, scene classification, CSV outputs, QC previews.

## §0 — Configuration

Everything tunable lives here. Nothing below this cell should contain a hard-coded path or
threshold.

In [2]:
from pathlib import Path

# ---------------------------------------------------------------- what we are reading
PROJECT   = Path("/home/jovyan/neve_ohw26")
SOURCE    = "s2"          # "s2" (development) | "planet" (Thursday)

S2_SAFE     = PROJECT / "S2C_MSIL2A_20250807T150731_N0511_R125_T20TNQ_20250807T193619.SAFE"
PLANET_GLOB = PROJECT / "data" / "*_AnalyticMS_SR_clip.tif"      # Thursday; not used yet

# ---------------------------------------------------------------- the analysis window
# Mirrors the real AOI geometry: a square box centred on the hydrophone. On S2 there is no
# hydrophone, so we drop a pretend one in the middle of a coastal window -- the range maths
# being exercised is identical.
#
# Leave WINDOW_CENTRE = None to read the whole scene (slow on a full S2 tile). Set it from the
# overview map in section 2.
WINDOW_CENTRE   = None        # (lon, lat) -- fill in from the overview in section 2
WINDOW_HALF_KM  = 5.0         # half-width; 5.0 -> a 10 x 10 km box, same as the Planet clip
HYDROPHONE      = None        # (lon, lat); None -> use the window centre

# ---------------------------------------------------------------- detector A (YOLO)
DEVICE     = "cpu"            # no GPU on this hub
SLICE      = 320              # tile size in px -- sweep {256, 320, 512}
IMGSZ      = 640              # network input size -- sweep {640, 1024}
CONF       = 0.10             # deliberately LOW: recall matters, precision is recovered later
IOU        = 0.5              # NMS threshold for merging duplicates across tile overlap
OVERLAP    = 0.2
BGR        = True             # ultralytics flips numpy arrays internally; verify with the sweep

# ---------------------------------------------------------------- vessel size filter
MIN_LENGTH_M = 0.0            # raise this once we know the noise floor in px
MAX_LENGTH_M = 20.0           # the small-vessel cut: AIS-dark craft

# ---------------------------------------------------------------- radiometry
# Sentinel-2 L2A, processing baseline >= 04.00, carries an additive offset that MUST be applied
# before scaling. This file is N0511, so it applies. Miss it and every reflectance is wrong by 0.1:
# water goes near-black, NDWI shifts, the NIR threshold silently misfires, and nothing errors.
S2_QUANT      = 10000.0
S2_BOA_OFFSET = -1000.0

# The Sentinel-2 L1C TCI recipe: DN8 = clip(reflectance * 10000 / 10, 0, 255).
# A FIXED linear scale, not a per-scene percentile stretch -- see plan section 5.
TCI_DIVISOR = 10.0

print(f"source={SOURCE}  slice={SLICE}  imgsz={IMGSZ}  conf={CONF}  device={DEVICE}")

source=s2  slice=320  imgsz=640  conf=0.1  device=cpu


## §1 — Imports, and what we reuse

`vessel_detect.py` already contains working, tested tiling / NMS / geometry. We import those
rather than restating them. The genuinely new and uncertain code — water mask, NIR detector,
fusion, dominance — is written in cells below where you can read and tune it.

In [3]:
import sys, re, glob, math, warnings
import numpy as np
import rasterio
from rasterio.windows import Window, from_bounds as window_from_bounds
from rasterio.warp import transform as rio_transform, transform_bounds
from rasterio.transform import xy as rio_xy
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)
sys.path.insert(0, str(PROJECT))

# --- reused, already working (see vessel_detect.py) --------------------------------
from vessel_detect import (
    load_model,        # cached HF download -> ultralytics YOLO
    predict_tiled,     # tile -> predict -> offset boxes back -> NMS
    iter_tiles,        # overlapping tile origins
    nms,               # IoU non-maximum suppression
    haversine_km,      # great-circle range
)

plt.rcParams["figure.dpi"] = 110
print("rasterio", rasterio.__version__, "| numpy", np.__version__)
print("reusing:", "load_model, predict_tiled, iter_tiles, nms, haversine_km")

rasterio 1.5.1 | numpy 2.5.2
reusing: load_model, predict_tiled, iter_tiles, nms, haversine_km


## §2 — What have we actually got?

The `.SAFE` download may still be in flight. This cell checks each band **opens and reads**,
not just that a file exists — a half-downloaded JP2 has a perfectly valid header and throws only
when you touch the pixels.

In [ ]:
S2_BANDS = {"blue": "B02", "green": "B03", "red": "B04", "nir": "B08", "tci": "TCI"}

def s2_band_paths(safe_dir):
    """Map logical band name -> path inside a .SAFE tree (10 m products only)."""
    r10 = list(Path(safe_dir).glob("GRANULE/*/IMG_DATA/R10m"))
    if not r10:
        raise FileNotFoundError(f"no GRANULE/*/IMG_DATA/R10m under {safe_dir}")
    out = {}
    for name, code in S2_BANDS.items():
        hits = list(r10[0].glob(f"*_{code}_10m.jp2"))
        if hits:
            out[name] = hits[0]
    return out


def check_readable(path, n=256):
    """Open and read a small window. Returns (ok, message).

    A truncated JP2 has a valid header, reports full width/height, and only fails when the
    codestream is touched -- so existence and shape are not evidence of a complete file.
    """
    try:
        with rasterio.open(path) as s:
            s.read(1, window=Window(s.width // 2, s.height // 2, n, n))
            return True, f"{s.width}x{s.height} @ {s.res[0]:g} m  {s.dtypes[0]}"
    except Exception as exc:
        return False, f"{type(exc).__name__}: {str(exc)[:70]}"


paths = s2_band_paths(S2_SAFE)
print(f"{'band':>6}  {'MB':>7}  status")
print("-" * 64)
ready = {}
for name in ("blue", "green", "red", "nir", "tci"):
    p = paths.get(name)
    if p is None:
        print(f"{name:>6}  {'--':>7}  NOT DOWNLOADED")
        continue
    ok, msg = check_readable(p)
    ready[name] = ok
    print(f"{name:>6}  {p.stat().st_size/1e6:7.1f}  {'OK  ' if ok else 'PARTIAL '}{msg}")

need = ["blue", "green", "red", "nir"]
missing = [b for b in need if not ready.get(b)]
print()
print("READY for the full pipeline" if not missing
      else f"still waiting on: {', '.join(missing)}  (re-run this cell)")

## §3 — Overview map: choose a window

A full S2 tile is 110 x 110 km. We do not want to run a tiled CPU detector over that, and the
real AOI is a 10 x 10 km clip anyway — so pick a coastal window with harbours or moorings in it.

This reads a **decimated** version of the blue band (cheap: rasterio subsamples on read rather
than loading 241 MB), and overlays a lon/lat grid so you can read a centre off the picture and
paste it into `WINDOW_CENTRE` in §0.

In [ ]:
def overview(path, max_px=1200, stretch=(2, 98)):
    """Decimated whole-scene read, percentile-stretched FOR DISPLAY ONLY.

    A percentile stretch is exactly the wrong transform for model input (see plan section 5) --
    it is fine here because nothing downstream consumes this array.
    """
    with rasterio.open(path) as s:
        f = max(1, int(max(s.width, s.height) / max_px))
        arr = s.read(1, out_shape=(1, s.height // f, s.width // f)).astype(np.float32)
        bounds, crs = s.bounds, s.crs
    lo, hi = np.percentile(arr[arr > 0], stretch) if (arr > 0).any() else (0, 1)
    return np.clip((arr - lo) / max(hi - lo, 1e-6), 0, 1), bounds, crs


img, bounds, crs = overview(paths["blue"])
w, s_, e, n = transform_bounds(crs, "EPSG:4326", *bounds)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(img, cmap="gray", extent=[w, e, s_, n], origin="upper")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title(f"{S2_SAFE.name.split('_')[5]}  —  pick a coastal window\n"
             f"dark = water, bright = land/cloud", fontsize=9)
ax.grid(alpha=0.3, ls=":")
plt.tight_layout(); plt.show()

print(f"scene covers lon {w:.3f}..{e:.3f}, lat {s_:.3f}..{n:.3f}")
print(f"a {2*WINDOW_HALF_KM:.0f} km box is ~{2*WINDOW_HALF_KM/111:.3f} deg of latitude")
print("\n--> read a centre off the map, set WINDOW_CENTRE in section 0, re-run it, continue")

## §4 — The loader seam

One `Scene` object, two loaders. Everything downstream touches only `Scene`, which is what makes
Thursday a config change.

`Scene.rgb8` is the model's input. `Scene.nir` / `Scene.green` are **reflectance** (0–1), used by
the water mask and detector B. `Scene.valid` marks real data — nodata outside a clip is 0, and a
0 that gets the L2A offset applied becomes −0.1, which would poison every statistic.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Scene:
    scene_id : str
    acq_utc  : str
    rgb8     : np.ndarray          # (H, W, 3) uint8  -- detector A input
    nir      : np.ndarray          # (H, W) float32 reflectance -- detector B input
    green    : np.ndarray          # (H, W) float32 reflectance -- NDWI
    valid    : np.ndarray          # (H, W) bool
    transform: object              # affine, for this window
    crs      : object
    res_m    : float
    tci8     : np.ndarray = None   # rendered RGB if the product ships one (S2 TCI / ortho_visual)
    meta     : dict = field(default_factory=dict)

    @property
    def shape(self): return self.valid.shape

    def to_lonlat(self, cols, rows):
        """Pixel centres -> (lon, lat). Wraps rasterio so callers never touch the CRS."""
        xs, ys = rio_xy(self.transform, list(rows), list(cols), offset="center")
        lon, lat = rio_transform(self.crs, "EPSG:4326", xs, ys)
        return np.asarray(lon), np.asarray(lat)


def _window_for(src, centre_lonlat, half_km):
    """Square window about a lon/lat, in the raster's own CRS."""
    if centre_lonlat is None:
        return None
    xs, ys = rio_transform("EPSG:4326", src.crs, [centre_lonlat[0]], [centre_lonlat[1]])
    cx, cy, half_m = xs[0], ys[0], half_km * 1000.0
    return window_from_bounds(cx - half_m, cy - half_m, cx + half_m, cy + half_m,
                              transform=src.transform)


def to_tci_rgb(red, green, blue, divisor=TCI_DIVISOR):
    """Reflectance (0-1) -> 8-bit RGB using the FIXED Sentinel-2 L1C TCI scaling.

    DN8 = clip(reflectance * 10000 / divisor, 0, 255).

    Deliberately NOT a per-scene percentile stretch: a stretch makes each scene's radiometry
    depend on its own content, so identical water renders differently from scene to scene. That
    is precisely the domain drift a pretrained model is chosen to avoid. See plan section 5.
    """
    stack = np.stack([red, green, blue], axis=-1) * (S2_QUANT / divisor)
    return np.clip(stack, 0, 255).astype(np.uint8)


def load_s2_safe(safe_dir, centre_lonlat=None, half_km=5.0):
    """Sentinel-2 L2A .SAFE -> Scene. Development path."""
    p = s2_band_paths(safe_dir)
    name = Path(safe_dir).name
    m = re.search(r"_(\d{8})T(\d{6})_", name)
    acq = (f"{m.group(1)[:4]}-{m.group(1)[4:6]}-{m.group(1)[6:8]}"
           f"T{m.group(2)[:2]}:{m.group(2)[2:4]}:{m.group(2)[4:6]}Z") if m else ""
    tile = re.search(r"_(T\d{2}[A-Z]{3})_", name)

    with rasterio.open(p["blue"]) as ref:
        win = _window_for(ref, centre_lonlat, half_km)
        tf  = ref.window_transform(win) if win is not None else ref.transform
        crs, res = ref.crs, abs(ref.transform.a)

    def band(key):
        with rasterio.open(p[key]) as s:
            return s.read(1, window=win).astype(np.float32)

    dn = {k: band(k) for k in ("blue", "green", "red", "nir")}
    # nodata is 0 -- capture it BEFORE the offset turns 0 into -0.1
    valid = np.logical_and.reduce([v > 0 for v in dn.values()])
    rho = {k: np.clip((v + S2_BOA_OFFSET) / S2_QUANT, 0, None) for k, v in dn.items()}
    for v in rho.values():
        v[~valid] = 0.0

    tci8 = None
    if p.get("tci") and check_readable(p["tci"])[0]:
        with rasterio.open(p["tci"]) as s:
            tci8 = s.read([1, 2, 3], window=win).transpose(1, 2, 0).astype(np.uint8)

    return Scene(
        scene_id=tile.group(1) + "_" + (m.group(1) if m else "") if tile else name[:32],
        acq_utc=acq,
        rgb8=to_tci_rgb(rho["red"], rho["green"], rho["blue"]),
        nir=rho["nir"], green=rho["green"], valid=valid,
        transform=tf, crs=crs, res_m=res, tci8=tci8,
        meta={"source": "s2_l2a", "safe": name, "boa_offset": S2_BOA_OFFSET},
    )

### The PlanetScope loader

Written now so the seam is real rather than aspirational, but **untested** — no imagery has been
delivered yet. Band order for `ortho_analytic_4b_sr` is Blue, Green, Red, NIR.

Two things to confirm against the first delivered scene on Thursday:
1. **The reflectance scale.** Planet SR is normally scaled by 10000 with **no additive offset**.
   Confirm from the delivered metadata rather than assuming.
2. **The right `divisor` for `to_tci_rgb`.** It sets how bright the derived RGB is. Section 5's
   acceptance test — derived RGB vs. `ortho_visual` on the same capture — is how you pick it.

In [ ]:
PLANET_QUANT      = 10000.0
PLANET_SR_OFFSET  = 0.0      # confirm from the delivered metadata

def load_planet_analytic(tif, centre_lonlat=None, half_km=5.0, visual_tif=None):
    """PlanetScope ortho_analytic_4b_sr -> Scene. Bands: 1=B 2=G 3=R 4=NIR.

    UNTESTED -- no delivered imagery yet. Verify against the first scene Thursday morning.
    """
    with rasterio.open(tif) as s:
        win = _window_for(s, centre_lonlat, half_km)
        tf  = s.window_transform(win) if win is not None else s.transform
        arr = s.read(window=win).astype(np.float32)      # (4, H, W)
        crs, res = s.crs, abs(s.transform.a)

    blue, green, red, nir = arr[0], arr[1], arr[2], arr[3]
    valid = np.logical_and.reduce([b > 0 for b in (blue, green, red, nir)])
    scale = lambda b: np.clip((b + PLANET_SR_OFFSET) / PLANET_QUANT, 0, None) * valid

    tci8 = None
    if visual_tif:
        with rasterio.open(visual_tif) as s:
            tci8 = s.read([1, 2, 3], window=win).transpose(1, 2, 0).astype(np.uint8)

    from vessel_detect import parse_scene_meta
    scene_id, acq = parse_scene_meta(str(tif))
    return Scene(scene_id=scene_id, acq_utc=acq,
                 rgb8=to_tci_rgb(scale(red), scale(green), scale(blue)),
                 nir=scale(nir), green=scale(green), valid=valid,
                 transform=tf, crs=crs, res_m=res, tci8=tci8,
                 meta={"source": "planet_analytic_sr", "path": str(tif)})
print("loader seam defined: load_s2_safe / load_planet_analytic -> Scene")

## §5 — Load the window

In [ ]:
if WINDOW_CENTRE is None:
    raise SystemExit("Set WINDOW_CENTRE in section 0 from the overview map, re-run it, then this.")

sc = load_s2_safe(S2_SAFE, WINDOW_CENTRE, WINDOW_HALF_KM) if SOURCE == "s2" else \
     load_planet_analytic(sorted(glob.glob(str(PLANET_GLOB)))[0], WINDOW_CENTRE, WINDOW_HALF_KM)

hydro = HYDROPHONE or WINDOW_CENTRE
h, w = sc.shape
print(f"{sc.scene_id}   {sc.acq_utc}")
print(f"{w} x {h} px @ {sc.res_m:g} m  =  {w*sc.res_m/1000:.1f} x {h*sc.res_m/1000:.1f} km")
print(f"valid {100*sc.valid.mean():.1f}%   |   pretend hydrophone at {hydro}")
print(f"water reflectance: NIR median {np.median(sc.nir[sc.valid]):.4f}  "
      f"green median {np.median(sc.green[sc.valid]):.4f}")

fig, ax = plt.subplots(1, 3, figsize=(14, 5))
ax[0].imshow(sc.rgb8);                      ax[0].set_title("derived RGB (fixed TCI scaling)")
ax[1].imshow(sc.tci8 if sc.tci8 is not None else sc.rgb8)
ax[1].set_title("shipped TCI" if sc.tci8 is not None else "(no shipped RGB)")
im = ax[2].imshow(sc.nir, cmap="magma", vmin=0, vmax=0.15)
ax[2].set_title("NIR reflectance — water is dark, boats glow")
plt.colorbar(im, ax=ax[2], fraction=0.046)
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## §6 — Detector A, and the §5 acceptance test

Run the model twice on the identical capture: once on the RGB we derived from reflectance, once
on the RGB the product ships. If they disagree, our scaling is wrong — and on Thursday that
failure would be indistinguishable from "the model doesn't work on PlanetScope".

Rehearsing it here, on a scene where both versions certainly exist, is the whole point of using
Sentinel-2 today.

In [ ]:
model, weights = load_model(DEVICE)
print("weights:", weights)

In [ ]:
import time

def run_detector_a(rgb8, valid, label):
    t0 = time.time()
    dets = predict_tiled(model, rgb8, valid, SLICE, IMGSZ, CONF,
                         overlap=OVERLAP, iou_thr=IOU, bgr=BGR)
    dt = time.time() - t0
    n_small = sum(1 for (x1, y1, x2, y2), _ in dets
                  if max(x2 - x1, y2 - y1) * sc.res_m <= MAX_LENGTH_M)
    print(f"{label:>16}: {len(dets):4d} raw  {n_small:4d} under {MAX_LENGTH_M:.0f} m   [{dt:.0f}s]")
    return dets

det_derived = run_detector_a(sc.rgb8, sc.valid, "derived RGB")
det_shipped = run_detector_a(sc.tci8, sc.valid, "shipped TCI") if sc.tci8 is not None else None

In [ ]:
def agreement(a, b, tol_px=3.0):
    """Fraction of `a` boxes with a `b` box centre within tol_px. Symmetric report."""
    if not a or not b:
        return float("nan"), float("nan")
    ca = np.array([[(x1+x2)/2, (y1+y2)/2] for (x1, y1, x2, y2), _ in a])
    cb = np.array([[(x1+x2)/2, (y1+y2)/2] for (x1, y1, x2, y2), _ in b])
    d  = np.linalg.norm(ca[:, None, :] - cb[None, :, :], axis=-1)
    return (d.min(axis=1) <= tol_px).mean(), (d.min(axis=0) <= tol_px).mean()

if det_shipped is not None:
    fwd, rev = agreement(det_derived, det_shipped)
    print(f"derived found in shipped : {fwd:6.1%}")
    print(f"shipped found in derived : {rev:6.1%}")
    print()
    if min(fwd, rev) > 0.9:
        print("PASS  -- scaling is sound. Same test on Planet Thursday decides analytic-only.")
    else:
        print("FAIL  -- tune TCI_DIVISOR in section 0 and re-run. Do NOT proceed on a failed\n"
              "        scaling: it is indistinguishable downstream from a model that does not\n"
              "        transfer, and you would draw the wrong conclusion from it.")
else:
    print("No shipped RGB available -- cannot run the acceptance test on this scene.")

In [ ]:
def show_dets(rgb8, dets, title, max_px=1400, color="#ff2828"):
    """Boxes drawn on the scene, padded so a 2-3 px vessel is actually visible."""
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(rgb8)
    for (x1, y1, x2, y2), s in dets:
        pad = 5
        ax.add_patch(plt.Rectangle((x1-pad, y1-pad), (x2-x1)+2*pad, (y2-y1)+2*pad,
                                   fill=False, ec=color, lw=1.0))
    ax.set_title(f"{title}  —  {len(dets)} detections @ conf>={CONF}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

show_dets(sc.rgb8, det_derived, "Detector A on derived RGB")

---
## Next stage (not yet written)

- **§7 water mask** — NDWI from green + NIR, threshold, erode ~3 px, combine with `valid`
- **§8 detector B** — adaptive NIR threshold (median + N·MAD, sweep N), connected components
  via `cv2.connectedComponentsWithStats`, area + compactness filters
- **§9 fusion** — match A↔B by distance; A∩B, A-only, B-only
- **§10 georeference + range** → `detections.csv` (+ `detector`, `nir_snr` columns)
- **§11 dominance** → `scenes.csv` (EMPTY / SINGLE-DOMINANT / CROWDED)
- **§12 QC** — side-by-side RGB and NIR chips for eyeball verification
- **§13 geometry self-test** — a synthetic point at a known lon/lat, assert the range is right

Stop and review before continuing.